In [1]:
%pip install scikit-learn

python(19369) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import pyarrow.parquet as pq

df_votes = pq.read_table("./data/votes.parquet").to_pandas()
print(" Votes dataset loaded successfully.")
df_conversations = pq.read_table("./data/conversations.parquet").to_pandas()
print(" Conversations dataset loaded successfully.")
df_reactions = pq.read_table("./data/reactions.parquet").to_pandas()
print(" Reactions dataset loaded successfully.")

 Votes dataset loaded successfully.


In [ ]:
df_conversations[df_conversations["conversation_pair_id"] == '8a15faf5c5374cc1bbbc84426a8bc0e6-e4ed6e033a844c9eb5eab207533896b6']

In [ ]:
df_votes['conversation_pair_id'][1]

In [ ]:
df_votes.columns.to_list()

<h2>Biais de longueur</h2>

In [ ]:
col_int = ['conversation_pair_id',
 'model_a_name',
 'model_b_name',
 'chosen_model_name',
 'opening_msg',
 'both_equal',
 'conversation_a',
 'conversation_b',
 'conv_turns',
 'session_hash',
 'visitor_id',
 'system_prompt_b',
 'system_prompt_a']

df_votes_int = df_votes[col_int]

In [ ]:
df_conv_filtre = df_conversations[["conversation_pair_id", "total_conv_a_output_tokens", "total_conv_b_output_tokens"]]

df_merge = df_votes_int.merge(df_conv_filtre, left_on="conversation_pair_id", right_on="conversation_pair_id", how="left")

df_merge.head()

In [ ]:
df_nan = df_merge[(df_merge['total_conv_a_output_tokens'] == 0) | (df_merge['total_conv_b_output_tokens'] == 0)]

df_without_null = df_merge[(df_merge['total_conv_a_output_tokens'] != 0) & (df_merge['total_conv_b_output_tokens'] != 0)]

In [ ]:
df_without_null = df_without_null.copy()
df_without_null["token_diff_a_minus_b"] = (
    df_without_null["total_conv_a_output_tokens"] - df_without_null["total_conv_b_output_tokens"]
)

df_corr = df_without_null[
    (df_without_null["chosen_model_name"] == df_without_null["model_a_name"])
    | (df_without_null["chosen_model_name"] == df_without_null["model_b_name"])
][ ["token_diff_a_minus_b", "chosen_model_name", "model_a_name"] ].copy()

df_corr["chosen_a"] = (df_corr["chosen_model_name"] == df_corr["model_a_name"]).astype(int)

correlation = df_corr["token_diff_a_minus_b"].corr(df_corr["chosen_a"])

print(f"Corrélation (diff tokens A-B vs choix de A): {correlation:.4f}")
print(f"Nombre d'observations utilisées: {len(df_corr)}")

Il y a un biais de longueur existant mais modéré. La faible valeur indique que sur les échantillons observés, une longueur plus importante de la réponse peut être favorisée, mais la relativement faible valeur montre que c'est loin d'être une raison prédominante.

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df_model = df_votes.merge(
    df_conv_filtre, on="conversation_pair_id", how="left"
).copy()

# Garder uniquement les duels avec choix explicite A ou B
df_model = df_model[
    (df_model["chosen_model_name"] == df_model["model_a_name"])
    | (df_model["chosen_model_name"] == df_model["model_b_name"])
]

# Variable d'intérêt
df_model["token_diff_a_minus_b"] = (
    df_model["total_conv_a_output_tokens"] - df_model["total_conv_b_output_tokens"]
)

# Cible
df_model["chosen_a"] = (
    df_model["chosen_model_name"] == df_model["model_a_name"]
).astype(int)

# Covariables numériques générales
df_model["opening_msg_len"] = df_model["opening_msg"].fillna("").str.len()
df_model["system_prompt_a_len"] = df_model["system_prompt_a"].fillna("").str.len()
df_model["system_prompt_b_len"] = df_model["system_prompt_b"].fillna("").str.len()
df_model["timestamp"] = pd.to_datetime(df_model["timestamp"], errors="coerce")
df_model["hour"] = df_model["timestamp"].dt.hour
df_model["dayofweek"] = df_model["timestamp"].dt.dayofweek
df_model["is_unedited_prompt_num"] = pd.to_numeric(
    df_model["is_unedited_prompt"], errors="coerce"
)

# Différences A-B sur signaux conversationnels potentiellement utiles
paired_metrics = [
    "conv_useful",
    "conv_creative",
    "conv_clear_formatting",
    "conv_incorrect",
    "conv_superficial",
    "conv_instructions_not_followed",
    "conv_complete",
]
for metric in paired_metrics:
    col_a = f"{metric}_a"
    col_b = f"{metric}_b"
    if col_a in df_model.columns and col_b in df_model.columns:
        a_num = pd.to_numeric(df_model[col_a].astype("float"), errors="coerce")
        b_num = pd.to_numeric(df_model[col_b].astype("float"), errors="coerce")
        df_model[f"{metric}_diff_a_minus_b"] = a_num - b_num

feature_num = [
    "token_diff_a_minus_b",
    "conv_turns",
    "opening_msg_len",
    "system_prompt_a_len",
    "system_prompt_b_len",
    "hour",
    "dayofweek",
    "is_unedited_prompt_num",
] + [
    c for c in df_model.columns if c.endswith("_diff_a_minus_b")
 and c != "token_diff_a_minus_b"
]

feature_cat = ["selected_category", "model_a_name", "model_b_name"]

X = df_model[feature_num + feature_cat]
y = df_model["chosen_a"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, feature_num),
        ("cat", cat_pipe, feature_cat),
    ]
)

logit = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=3000, solver="saga")),
    ]
)

logit.fit(X_train, y_train)
y_pred = logit.predict(X_test)
y_proba = logit.predict_proba(X_test)[:, 1]

feature_names = logit.named_steps["preprocess"].get_feature_names_out()
coef_all = logit.named_steps["model"].coef_[0]
coef_token_diff = coef_all[np.where(feature_names == "num__token_diff_a_minus_b")[0][0]]

coef_table = pd.DataFrame({"feature": feature_names, "coef": coef_all})
top_coef = coef_table.reindex(coef_table["coef"].abs().sort_values(ascending=False).index).head(10)

print(f"Coefficient ajusté token_diff_a_minus_b: {coef_token_diff:.6f}")
print(f"Odds ratio ajusté pour +100 tokens: {float(np.exp(100 * coef_token_diff)):.4f}")
print(f"Accuracy test: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC test: {roc_auc_score(y_test, y_proba):.4f}")
print(f"N observations: {len(df_model)}")
print("\nTop 10 coefficients (|coef|):")
print(top_coef.to_string(index=False))

In [ ]:
from sklearn.linear_model import LogisticRegression

# Échantillon commun pour comparer "avant" vs "après correction"
df_bt = df_votes.merge(df_conv_filtre, on="conversation_pair_id", how="left").copy()
df_bt = df_bt[
    (df_bt["chosen_model_name"] == df_bt["model_a_name"])
    | (df_bt["chosen_model_name"] == df_bt["model_b_name"])
]
df_bt = df_bt.dropna(subset=["total_conv_a_output_tokens", "total_conv_b_output_tokens", "model_a_name", "model_b_name"]).copy()
df_bt["token_diff_a_minus_b"] = df_bt["total_conv_a_output_tokens"] - df_bt["total_conv_b_output_tokens"]
df_bt["chosen_a"] = (df_bt["chosen_model_name"] == df_bt["model_a_name"]).astype(int)

models = sorted(set(df_bt["model_a_name"]).union(set(df_bt["model_b_name"])))
ref_model = models[-1]
model_to_col = {m: i for i, m in enumerate([m for m in models if m != ref_model])}

n = len(df_bt)
p = len(models) - 1
X_bt = np.zeros((n, p), dtype=float)

idx_a = df_bt["model_a_name"].map(model_to_col)
idx_b = df_bt["model_b_name"].map(model_to_col)
rows = np.arange(n)

mask_a = idx_a.notna()
mask_b = idx_b.notna()
X_bt[rows[mask_a], idx_a[mask_a].astype(int)] = 1.0
X_bt[rows[mask_b], idx_b[mask_b].astype(int)] = -1.0

y_bt = df_bt["chosen_a"].to_numpy()

# Bradley-Terry non corrigé
bt_base = LogisticRegression(fit_intercept=False, max_iter=3000, solver="lbfgs", C=1e6)
bt_base.fit(X_bt, y_bt)

# Bradley-Terry corrigé par la différence de longueur
X_bt_corr = np.column_stack([X_bt, df_bt["token_diff_a_minus_b"].to_numpy()])
bt_corr = LogisticRegression(fit_intercept=False, max_iter=3000, solver="lbfgs", C=1e6)
bt_corr.fit(X_bt_corr, y_bt)

coef_base = bt_base.coef_[0]
coef_corr = bt_corr.coef_[0][:p]
coef_token = bt_corr.coef_[0][p]

score_base = {m: 0.0 for m in models}
score_corr = {m: 0.0 for m in models}
for m, j in model_to_col.items():
    score_base[m] = coef_base[j]
    score_corr[m] = coef_corr[j]

rank_base = pd.DataFrame(score_base.items(), columns=["model", "score_base"]).sort_values("score_base", ascending=False)
rank_base["rank_base"] = np.arange(1, len(rank_base) + 1)
rank_corr = pd.DataFrame(score_corr.items(), columns=["model", "score_corr"]).sort_values("score_corr", ascending=False)
rank_corr["rank_corr"] = np.arange(1, len(rank_corr) + 1)

rank_compare = rank_base.merge(rank_corr, on="model")
rank_compare["delta_rank"] = rank_compare["rank_base"] - rank_compare["rank_corr"]

rho_rank = rank_compare[["rank_base", "rank_corr"]].corr(method="spearman").loc["rank_base", "rank_corr"]

print(f"Coefficient correction longueur (token_diff_a_minus_b): {coef_token:.6f}")
print(f"Spearman des rangs avant/après correction: {rho_rank:.4f}")
print("\nTop modèles qui montent après correction (delta_rank > 0):")
print(
    rank_compare[rank_compare["delta_rank"] > 0]
    .sort_values("delta_rank", ascending=False)
    .head(10)
    .to_string(index=False)
)
print("\nTop modèles qui baissent après correction (delta_rank < 0):")
print(
    rank_compare[rank_compare["delta_rank"] < 0]
    .sort_values("delta_rank", ascending=True)
    .head(10)
    .to_string(index=False)
 )

### Synthèse (biais de longueur et classement Bradley-Terry)
- La différence de longueur (`tokens_A - tokens_B`) est positivement associée au choix de A (corrélation positive mais faible).
- En régression logistique multivariée, l’effet ajusté reste positif : +100 tokens pour A augmente ses odds d’être choisi d’environ 2 à 2,5 %.
- Le biais de longueur est donc réel, mais d’ampleur modérée au niveau individuel.
- Après correction de longueur dans le modèle Bradley-Terry, le classement global reste proche de l’original (Spearman ≈ 0,98).
- Néanmoins, certains modèles changent fortement de rang : la correction affecte surtout des cas particuliers plutôt que l’ordre global.
- Interprétation : une partie des écarts de préférence observés provient de la longueur des réponses, mais ce n’est pas le facteur dominant.

<h2>Biais de position</h2>

In [ ]:
from scipy.stats import chi2_contingency
import numpy as np

def normalize_liked(series):
    s = series.astype(str).str.strip().str.lower()
    mapping = {
        "true": "true",
        "false": "false",
        "1": "true",
        "0": "false",
        "yes": "true",
        "no": "false",
    }
    return s.map(mapping)

def chi2_position_effect(df, label="global"):
    sub = df.dropna(subset=["model_pos", "liked"]).copy()
    sub["model_pos"] = sub["model_pos"].astype(str).str.strip().str.lower()
    sub = sub[sub["model_pos"].isin(["a", "b"])]
    sub["liked_norm"] = normalize_liked(sub["liked"])

    valid = sub.dropna(subset=["liked_norm"]).copy()

    contingency = pd.crosstab(valid["model_pos"], valid["liked_norm"])
    for col in ["false", "true"]:
        if col not in contingency.columns:
            contingency[col] = 0
    contingency = contingency[["false", "true"]].reindex(["a", "b"], fill_value=0)

    n = int(contingency.to_numpy().sum())
    r, c = contingency.shape
    rate_a = contingency.loc["a", "true"] / contingency.loc["a"].sum() if contingency.loc["a"].sum() > 0 else np.nan
    rate_b = contingency.loc["b", "true"] / contingency.loc["b"].sum() if contingency.loc["b"].sum() > 0 else np.nan

    if n == 0 or (contingency.sum(axis=1) == 0).any() or (contingency.sum(axis=0) == 0).any():
        return {
            "subset": label,
            "n": n,
            "liked_rate_a": rate_a,
            "liked_rate_b": rate_b,
            "diff_a_minus_b": rate_a - rate_b if pd.notna(rate_a) and pd.notna(rate_b) else np.nan,
            "chi2": np.nan,
            "p_value": np.nan,
            "cramers_v": np.nan,
            "contingency": contingency,
            "model_pos_counts": valid["model_pos"].value_counts(dropna=False),
            "liked_raw_top": sub["liked"].astype(str).str.lower().value_counts(dropna=False).head(5),
        }

    chi2, p_value, dof, expected = chi2_contingency(contingency)
    cramer_v = np.sqrt(chi2 / (n * (min(r, c) - 1))) if min(r, c) > 1 else np.nan

    return {
        "subset": label,
        "n": n,
        "liked_rate_a": rate_a,
        "liked_rate_b": rate_b,
        "diff_a_minus_b": rate_a - rate_b if pd.notna(rate_a) and pd.notna(rate_b) else np.nan,
        "chi2": chi2,
        "p_value": p_value,
        "cramers_v": cramer_v,
        "contingency": contingency,
        "model_pos_counts": valid["model_pos"].value_counts(dropna=False),
        "liked_raw_top": sub["liked"].astype(str).str.lower().value_counts(dropna=False).head(5),
    }

results = [chi2_position_effect(df_reactions, label="global")]

for qualif in ["creative", "useful", "incorrect"]:
    if qualif in df_reactions.columns:
        df_q = df_reactions[df_reactions[qualif] == True]
        results.append(chi2_position_effect(df_q, label=qualif))
    else:
        print(f"Colonne absente dans df_reactions: {qualif}")

summary = pd.DataFrame([{k: v for k, v in r.items() if k not in ["contingency", "model_pos_counts", "liked_raw_top"]} for r in results])
summary["significatif_5pct"] = summary["p_value"] < 0.05

print("Résumé des tests chi-deux (liked ~ model_pos):")
display(summary)

for r in results:
    print(f"\n--- Distribution model_pos (valid liked): {r['subset']} ---")
    display(r["model_pos_counts"].rename("count").to_frame())
    print(f"--- Contingence: {r['subset']} ---")
    display(r["contingency"])

    print(f"--- Top valeurs brutes de liked: {r['subset']} ---")
    display(r["liked_raw_top"].rename("count").to_frame())

### Synthèse (biais de position)
- Le test du chi-deux global (`liked ~ model_pos`) est significatif : $\chi^2=9.10$, $p=0.0026$ (N=94 294).
- L'effet de position reste toutefois très faible en taille d’effet (Cramér’s $V=0.0098$).
- Taux de réactions positives : position A = 67.41 %, position B = 68.33 % (écart = -0.92 point).
- Pour les sous-ensembles `creative` et `useful`, l’effet n’est pas significatif (p=0.438 et p=0.953).
- Pour `incorrect`, le test est non identifiable car `liked` est constant (100 % `false`).
- Conclusion : il existe un biais de position détectable globalement, mais son impact pratique est faible.

## Conclusion — protocole d’annotation compar:IA v2
**Objectif** : réduire simultanément les biais de position, de longueur et de sélection, avec contrainte produit $\Delta$ abandon $\leq 15\%$ (vs version actuelle).

### 1) Design d’interface (réduction des biais sans coût cognitif fort)
- **Masquage systématique des identités** jusqu’au vote, comme dans Chatbot Arena, pour limiter les effets de marque et d’attentes utilisateur (Chiang et al., 2024 ; LMSYS Arena blog, 2023).
- **Randomisation 50/50 de la position A/B** à chaque duel + journalisation de `model_pos` pour correction ex post des biais de position (Zheng et al., 2023 signale explicitement les biais de position/verbosity en jugement pairwise).
- **Affichage progressif des réponses longues** (aperçu + “voir plus”) symétrique pour A et B, afin de réduire l’avantage perceptif des réponses longues sans tronquer l’information.
- **Vote en 1 clic + option “égalité / les deux mauvais”** pour garder une friction minimale (préserve le taux de complétion).

### 2) Échantillonnage et sélection des matchs (biais de sélection)
- **Plan mixte de sampling** : 70% paires informatives (incertitude élevée de rating) + 30% paires d’exploration uniforme pour couvrir tous les modèles.
- **Quota dynamique par langue / catégorie de prompt** pour éviter qu’un sous-groupe d’utilisateurs domine le leaderboard (Chiang et al., 2024 insiste sur la diversité des prompts crowdsourcés).
- **Limite de répétition par session** (pas de rematch immédiat même paire) pour réduire les dépendances comportementales.

### 3) Modèle statistique de ranking corrigé
- Estimer le score modèle avec un **Bradley-Terry logistique** incluant covariables :
  $$P(A\succ B)=\sigma\left((\theta_A-\theta_B)+\beta_{pos}\,I[A\text{ à gauche}]+\beta_{len}(len_A-len_B)+\gamma^\top z\right)$$
- `z` inclut au minimum : langue, catégorie, longueur du prompt, tour de conversation.
- Publier **deux classements** : brut et corrigé (transparence méthodologique), pratique cohérente avec l’esprit “arena + robustesse statistique” (Chiang et al., 2024).

### 4) Garde-fous produit (contrainte abandon ≤ 15%)
- **A/B test séquentiel** : v1 (contrôle) vs v2 (protocole proposé).
- KPI principal : `abandon_rate` (session sans vote). Condition d’acceptation :
  $$\frac{abandon_{v2}-abandon_{v1}}{abandon_{v1}} \le 0.15$$
- KPI secondaires : temps médian au vote, nombre de tours avant vote, taux d’égalité.
- **Rollout progressif** (10% → 25% → 50% → 100%) avec arrêt automatique si dépassement du seuil de 15%.

### 5) Pourquoi ce protocole est crédible
- Le cadre pairwise anonyme + randomisé est validé à grande échelle par Chatbot Arena pour produire des préférences exploitables (Chiang et al., 2024).
- Les biais de **position** et de **verbosity/longueur** sont documentés dans la littérature arena/LLM-judge, et motivent l’ajout explicite de covariables et de randomisation (Zheng et al., 2023).
- Le compromis UX (vote simple, changements invisibles côté effort) vise précisément à contrôler l’abandon tout en améliorant la validité des annotations.

**Références (arena)**
- Chiang, W.-L. et al. (2024). *Chatbot Arena: An Open Platform for Evaluating LLMs by Human Preference*. arXiv:2403.04132.
- Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*. arXiv:2306.05685.
- LMSYS (2023). *Chatbot Arena: Benchmarking LLMs in the Wild with Elo Ratings* (blog technique).

# Partie 8 : Validité de construit

La validité de construit du label `creative` dans compar:IA doit être vérifiée pour s'assurer qu'il mesure bien la créativité réelle, et non des biais comme la longueur, le style ou la clarté rédactionnelle. Nous allons donc procéder à des tests de validité convergente et discriminante, puis discuter des limites théoriques du jugement humain de la créativité.

## 2. Validité convergente

Nous testons si `creative` corrèle positivement avec des métriques objectives de créativité :
- MATTR (Famille A) appliqué au champ response_content de comparia-reactions
- N-gramme Rarity Score calculé sur les réponses vs. un corpus de référence français
- Distance d’embedding entre response_content et question_content

In [ ]:
# •	MATTR (Famille A) appliqué au champ response_content de comparia-reactions
def mattr(text, window=50):
    tokens = str(text).split()
    if len(tokens) < window:
        return len(set(tokens)) / max(1, len(tokens))
    scores = []
    for i in range(len(tokens) - window + 1):
        window_tokens = tokens[i:i+window]
        scores.append(len(set(window_tokens)) / window)
    return np.mean(scores)

df_reactions['mattr'] = df_reactions['response_content'].apply(lambda x: mattr(x, window=50))

In [ ]:
# •	N-gramme Rarity Score calculé sur les réponses vs. un corpus de référence français

from datasets import load_dataset
import numpy as np
from itertools import islice

import re

def clean_text(text):
    text = re.sub(r"\*\*", "", text)
    text = re.sub(r"[^\w\s']", " ", text)
    return text

# Corpus de réference 
dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.fr",
    split="train",
    streaming=True
)

reference_texts = [x["text"] for x in islice(dataset, 50000)]

print(f"{len(reference_texts):,} textes chargés")

def extract_bigrams(text):
    tokens = clean_text(str(text)).lower().split()
    return list(zip(tokens[:-1], tokens[1:]))

corpus_bigrams = set()
for text in reference_texts:
    corpus_bigrams.update(extract_bigrams(text))

print("Exemples de bigrammes du corpus")
sample_bigrams = list(corpus_bigrams)[:5]
for bg in sample_bigrams:
    print(f"  {bg[0]}, {bg[1]}")

print(f"Total bigrammes uniques : {len(corpus_bigrams):,}")

example_text = df_reactions['response_content'].dropna().iloc[0][:200]
example_tokens = clean_text(example_text).lower().split()[:10]
example_bigrams = list(zip(example_tokens[:-1], example_tokens[1:]))
print(f"Tokens : {example_tokens}")
print(f"Bigrammes : {example_bigrams}")
print(f"Absents du corpus : {[bg for bg in example_bigrams if bg not in corpus_bigrams][0:5]}")
print(f"Présents dans le corpus : {[bg for bg in example_bigrams if bg in corpus_bigrams][0:5]}")

# N-gram Rarity Score
def ngram_rarity_score(text, reference_set):
    tokens = clean_text(str(text)).lower().split()
    
    if len(tokens) < 2:
        return np.nan
        
    bigrams = list(zip(tokens[:-1], tokens[1:]))
    absent_count = sum(1 for bg in bigrams if bg not in reference_set)
    
    return absent_count / len(bigrams)


df_reactions["ngram_rarity"] = df_reactions["response_content"].apply(
    lambda x: ngram_rarity_score(x, corpus_bigrams)
)

print(f"Moyenne : {df_reactions['ngram_rarity'].mean():.4f}")
cor_ngram = df_reactions[['creative','ngram_rarity']].corr().iloc[0,1]
print(f"\nCorrélation creative / ngram rarity : {cor_ngram:.3f}")

In [ ]:
# # •	Distance d’embedding entre response_content et centroide des réponses

# from sklearn.metrics.pairwise import cosine_similarity
# from sentence_transformers import SentenceTransformer
# import numpy as np

# model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# # Centroide

# sample_for_centroid = df_reactions['response_content'].dropna().sample(
#     n=min(3000, len(df_reactions)),
#     random_state=42
# ).tolist()

# corpus_embeddings = model.encode(
#     sample_for_centroid,
#     batch_size=64,
#     show_progress_bar=True
# )

# corpus_centroid = corpus_embeddings.mean(axis=0)

# # Encoder les réponses

# texts = df_reactions['response_content'].fillna("").tolist()

# all_embeddings = model.encode(
#     texts,
#     batch_size=64,
#     show_progress_bar=True
# )

# #Embedding distance
# cos_similarities = cosine_similarity(all_embeddings, [corpus_centroid]).flatten()
# embedding_distances = 1 - cos_similarities

# df_reactions['embedding_distance'] = embedding_distances

# cor_embed = df_reactions[['creative', 'embedding_distance']].corr().iloc[0,1]

# print(f"Moyenne : {df_reactions['embedding_distance'].mean():.4f}")
# print(f"Corrélation creative / embedding_distance : {cor_embed:.3f}")


# • Distance d’embedding entre response_content et question_content

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

questions = df_reactions['question_content'].fillna("").tolist()
responses = df_reactions['response_content'].fillna("").tolist()

# On encode
question_embeddings = model.encode(
    questions,
    batch_size=64,
    show_progress_bar=True
)

response_embeddings = model.encode(
    responses,
    batch_size=64,
    show_progress_bar=True
)

# Similarité
cos_similarities = [
    cosine_similarity(
        question_embeddings[i].reshape(1, -1),
        response_embeddings[i].reshape(1, -1)
    )[0][0]
    for i in range(len(question_embeddings))
]

embedding_distances = 1 - np.array(cos_similarities)
df_reactions['embedding_distance_qr'] = embedding_distances
cor_embed = df_reactions[['creative', 'embedding_distance_qr']].corr().iloc[0,1]

print(f"Moyenne : {df_reactions['embedding_distance_qr'].mean():.4f}")
print(f"Corrélation creative / embedding_distance_qr : {cor_embed:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cor_ngram = df_reactions[['creative', 'ngram_rarity']].corr().iloc[0,1]
print(f"Corrélation creative/N-gram rarity : {cor_ngram:.3f}")

cor_mattr = df_reactions[['creative', 'mattr']].corr().iloc[0,1]
print(f"Corrélation creative/MATTR : {cor_mattr:.3f}")


metrics = ['creative','mattr','ngram_rarity','embedding_distance_qr']

plt.figure(figsize=(6,4))
sns.heatmap(df_reactions[metrics].corr(), annot=True, cmap='coolwarm')
plt.title("Corrélations validité convergente")
plt.show()

plt.figure(figsize=(5,3))
sns.scatterplot(x='mattr', y='creative', data=df_reactions, alpha=0.2)
plt.title('Creative vs MATTR')
plt.show()

sns.scatterplot(x='ngram_rarity', y='creative', data=df_reactions, alpha=0.2)  
plt.title('Creative vs N-gram Rarity')
plt.show()

sns.scatterplot(x='embedding_distance_qr', y='creative', data=df_reactions, alpha=0.2)
plt.title('Creative vs Embedding Distance')
plt.show()

Pour les métriques de créativité convergente, nous avons des corrélations positives mais très faibles r<<0.4, ce qui suggère que le label `creative` ne capture que très partiellement des aspects de créativité mesurés par ces métriques.

Ainsi le label `creative` ne reflète pas 
- La diversité lexicale
- La rareté statistique des bigrammes
- L'éloignement sémantique question-réponse
- La longueur de la réponse

## 3. Validité discriminante

Nous vérifions que `creative` ne corrèle pas avec des dimensions non créatives :
- Longueur de la réponse
- Clarté du formatage
- Correction factuelle

In [ ]:
# Corrélations discriminantes

df_reactions['response_length'] = df_reactions['response_content'].fillna("").str.len()
cor_length = df_reactions[['creative', 'response_length']].corr().iloc[0,1]
cor_format = df_reactions[['creative', 'clear_formatting']].corr().iloc[0,1]
cor_incorrect = df_reactions[['creative', 'incorrect']].corr().iloc[0,1]
cor_complete = df_reactions[['creative', 'complete']].corr().iloc[0,1]

print(f"Corrélation creative/longueur : {cor_length:.3f}")
print(f"Corrélation creative/clear_formatting : {cor_format:.3f}")
print(f"Corrélation creative/incorrect : {cor_incorrect:.3f}")
print(f"Corrélation creative/complete : {cor_complete:.3f}")

plt.figure(figsize=(6,4))
sns.heatmap(df_reactions[['creative', 'response_length', 'clear_formatting', 'incorrect', 'complete']].corr(), annot=True, cmap='coolwarm')
plt.title("Corrélations validité discriminante")
plt.show()

plt.figure(figsize=(5,3))
sns.scatterplot(x='response_length', y='creative', data=df_reactions, alpha=0.2)
plt.title('Creative vs Longueur')
plt.show()


- Corrélation creative/longueur : 0.052
La longueur de la réponse n’influence quasiment pas la probabilité d’être jugée créative.

- Corrélation creative/clear_formatting : 0.122
Les réponses bien formatées sont un peu plus souvent jugées créatives, effet restant faible.

- Corrélation creative/incorrect : -0.094 
La créativité perçue suppose une certaine validité du contenu. Les réponses jugées incorrectes sont un peu moins souvent jugées créatives, effet restant très faible.

- Corrélation creative/complete : 0.108  
Les réponses jugées complètes sont un peu plus souvent jugées créatives, effet restant faible.

Conclusion : Toutes ces corrélations sont faibles (< 0.15), ce qui indique que le label `creative` n’est pas confondu avec ces dimensions non créatives. Cela soutient la validité discriminante du label dans ce jeu de données.


## 4.3 Le paradoxe du juge dans le contexte compar:IA 

Problème : Comment juger de la créativité dans ce contexte ?
Une réponse vraiment nouvelle ne peut être reconnue comme créative que si l’utilisateur possède les structures cognitives adéquates.
Ex : une idée radicale peut sembler banale à un expert mais révolutionnaire pour un novice.
Cela crée un biais potentiel dans compar:IA.

**Forces du panel grand public**
- Diversité des profils et des attentes
- Volume de données, robustesse statistique
- Validité écologique (représente l’usage réel)

**Faiblesses**
- Jugements subjectifs, parfois superficiels
- Niveau de compréhension hétérogène
- Biais culturels ou linguistiques

Un panel grand public permet de mesurer la créativité telle qu’elle est perçue dans l’usage réel, mais peut sous-évaluer des idées très originales ou techniques. Un panel d’experts apporte une évaluation plus fine, mais moins représentative de l’usage courant. L’idéal est de combiner les deux approches pour une mesure robuste de la créativité.

# Conclusion

- **Enjeux épistémiques** : Mesurer la créativité des LLMs est un défi fondamental. La créativité est un concept complexe, multidimensionnel, et dépend du contexte, du public et des critères d’évaluation. Les LLMs peuvent produire des réponses originales, mais la reconnaissance de cette créativité dépend aussi des juges humains, de leurs attentes et de leur compréhension.
- **Métriques quantitatives** : Nous avons testé plusieurs métriques : diversité lexicale (MATTR), rareté des n-grammes, distance sémantique (SBERT), longueur, clarté du formatage, correction factuelle, complétude. Chacune capture une facette différente : aucune ne suffit seule, mais leur combinaison permet d’approcher la créativité perçue.
- **Biais identifiés et corrigés** : Nous avons quantifié des biais de longueur, de position, de formatage, et montré qu’ils existent mais restent modérés dans compar:IA. Les analyses discriminantes montrent que le label `creative` n’est pas confondu avec ces dimensions, ce qui soutient sa validité.
- **Limites** : Les corrélations entre le label `creative` et les métriques objectives sont faibles : la créativité perçue ne se réduit pas à la diversité lexicale ou à la rareté statistique. Le paradoxe du juge reste central : un panel grand public apporte une validité écologique, mais peut sous-évaluer l’originalité radicale. Un panel d’experts serait plus cohérent, mais moins représentatif.
- **Démarche basée sur les données** : Manipuler les données permet de quantifier les biais, de tester la robustesse des métriques, et d’objectiver les débats. Mais l’interprétation reste ouverte : la créativité n’est pas un score unique, et la mesure dépend du protocole, du public et des usages visés.